# NutriVision AI — Training Results & HP Analysis
Visualise training curves, HP search landscape, and model comparison.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import optuna

from configs.config import LOGS_DIR
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Notebooks ready.')

## 1. Training Curves

In [ ]:
history_files = list(LOGS_DIR.glob('*_history.json'))
if not history_files:
    print('No history found. Generating synthetic demo curves...')
    np.random.seed(42)
    epochs = list(range(1, 31))
    history = []
    for ep in epochs:
        history.append({
            'epoch': ep,
            'train_loss': max(0.1, 2.8 * 0.87**ep + np.random.uniform(-0.05, 0.05)),
            'val_loss':   max(0.15, 3.0 * 0.88**ep + np.random.uniform(-0.1, 0.1)),
            'train_acc':  min(95, 20 + ep*2.5 + np.random.uniform(-1, 1)),
            'val_acc':    min(88, 15 + ep*2.3 + np.random.uniform(-1.5, 1.5)),
            'val_top5':   min(98, 40 + ep*1.9 + np.random.uniform(-0.5, 0.5)),
            'val_f1':     min(0.88, 0.10 + ep*0.026 + np.random.uniform(-0.01, 0.01)),
            'lr':         max(1e-6, 1e-3 * 0.92**ep),
        })
    df = pd.DataFrame(history)
else:
    with open(history_files[-1]) as f:
        history = json.load(f)
    df = pd.DataFrame(history)

print(f'Epochs: {len(df)}  |  Best Val Acc: {df["val_acc"].max():.2f}%')

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Loss
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(df['epoch'], df['train_loss'], 'o-', color='#3b82f6', label='Train Loss', linewidth=2, markersize=4)
ax1.plot(df['epoch'], df['val_loss'],   's--', color='#ef4444', label='Val Loss',   linewidth=2, markersize=4)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Accuracy
ax2 = fig.add_subplot(gs[1, :2])
ax2.plot(df['epoch'], df['train_acc'], 'o-', color='#3b82f6', label='Train Acc',  linewidth=2, markersize=4)
ax2.plot(df['epoch'], df['val_acc'],   's--', color='#ef4444', label='Val Acc',    linewidth=2, markersize=4)
ax2.plot(df['epoch'], df['val_top5'],  '^-.', color='#22c55e', label='Top-5 Acc',  linewidth=2, markersize=4)
ax2.axhline(85, color='#94a3b8', linestyle=':', linewidth=1.5, label='Target 85%')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

# F1
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(df['epoch'], df['val_f1'], 'D-', color='#8b5cf6', linewidth=2, markersize=5)
ax3.axhline(0.82, color='#94a3b8', linestyle=':', linewidth=1.5, label='Target 0.82')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('Macro F1')
ax3.set_title('Validation Macro F1', fontsize=12, fontweight='bold')
ax3.legend(); ax3.grid(True, alpha=0.3)

# LR
ax4 = fig.add_subplot(gs[1, 2])
ax4.semilogy(df['epoch'], df['lr'], 'P-', color='#f59e0b', linewidth=2, markersize=5)
ax4.set_xlabel('Epoch'); ax4.set_ylabel('Learning Rate (log scale)')
ax4.set_title('LR Schedule', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.suptitle('NutriVision AI — Training Diagnostics', fontsize=15, fontweight='bold', y=1.01)
plt.savefig(LOGS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best epoch: {df.loc[df["val_acc"].idxmax(), "epoch"]}  |  Best val acc: {df["val_acc"].max():.2f}%')

## 2. Model Selection Comparison

In [ ]:
sel_path = LOGS_DIR / 'model_selection.json'
if sel_path.exists():
    with open(sel_path) as f:
        sel = json.load(f)
    rows = [{'Architecture': k, 'Val Accuracy (%)': v['val_acc']} for k, v in sel['results'].items()]
else:
    # Demo
    rows = [
        {'Architecture': 'efficientnet_b0',       'Val Accuracy (%)': 72.4},
        {'Architecture': 'resnet50',               'Val Accuracy (%)': 70.1},
        {'Architecture': 'mobilenetv3_large_100',  'Val Accuracy (%)': 68.7},
    ]

sel_df = pd.DataFrame(rows).sort_values('Val Accuracy (%)', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#22c55e' if i==0 else '#3b82f6' for i in range(len(sel_df))]
bars = ax.barh(sel_df['Architecture'], sel_df['Val Accuracy (%)'], color=colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, sel_df['Val Accuracy (%)']):
    ax.text(val+0.2, bar.get_y()+bar.get_height()/2, f'{val:.2f}%', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Validation Accuracy (%) — 3-epoch quick eval', fontsize=11)
ax.set_title('Model Selection: Architecture Comparison', fontsize=13, fontweight='bold')
ax.set_xlim(0, 100)
bars[0].set_label('🏆 Winner')
ax.legend(fontsize=10)
ax.grid(True, axis='x', alpha=0.25)
plt.tight_layout()
plt.savefig(LOGS_DIR / 'model_selection_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Hyperparameter Search Analysis

In [ ]:
hp_files = list(LOGS_DIR.glob('hp_search_*.json'))
if hp_files:
    with open(hp_files[0]) as f:
        hp = json.load(f)
    print('Best HP params:')
    for k, v in hp['best_params'].items():
        print(f'  {k}: {v}')
    print(f'  → Best val acc: {hp["best_val_acc"]:.2f}%')
    print(f'  → Completed: {hp["completed"]} | Pruned: {hp["pruned"]}')
else:
    print('No HP search results found (run train.py --mode full to generate).')
    print('Showing demo results:')
    demo_params = {
        'lr': 3.2e-4, 'weight_decay': 5e-5,
        'dropout': 0.4, 'hidden_dim': 512,
        'optimizer': 'adamw', 'scheduler': 'cosine'
    }
    for k, v in demo_params.items():
        print(f'  {k}: {v}')

In [ ]:
# Optuna visualisation (if study DB available)
optuna_db = LOGS_DIR / 'optuna_study.db'
if optuna_db.exists():
    try:
        study = optuna.load_study(
            study_name='food_hp_search',
            storage=f'sqlite:///{optuna_db}'
        )
        trials_df = study.trials_dataframe()
        print(f'Total trials: {len(trials_df)}')
        print(trials_df[['number','value','state']].head(10))

        # Optuna plot — optimization history
        from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
        fig1, ax1 = plt.subplots(figsize=(10, 4))
        plot_optimization_history(study, ax=ax1)
        ax1.set_title('HP Search: Optimization History', fontsize=13)
        plt.tight_layout()
        plt.savefig(LOGS_DIR / 'hp_optimization_history.png', dpi=150)
        plt.show()

        fig2, ax2 = plt.subplots(figsize=(8, 5))
        plot_param_importances(study, ax=ax2)
        ax2.set_title('HP Importance', fontsize=13)
        plt.tight_layout()
        plt.savefig(LOGS_DIR / 'hp_param_importances.png', dpi=150)
        plt.show()
    except Exception as e:
        print(f'Could not load Optuna study: {e}')
else:
    print('Optuna study DB not found. Run train.py --mode full to generate.')